In [1]:
# 1) Parar la sesión actual si existe
try:
    spark.stop()
except NameError:
    pass

In [2]:


# 2) Forzar Java 17 en este kernel
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# 3) Verifica que ahora ve Java 17
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)


In [3]:
# 4) Crear SparkSession nuevo
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .master("local[*]")
    .appName("test")
    .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 16:53:40 WARN Utils: Your hostname, codespaces-29ba17, resolves to a loopback address: 127.0.0.1; using 10.0.12.144 instead (on interface eth0)
26/03/07 16:53:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 16:53:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:

# 5) Leer directo el gzip (no hace falta descomprimir)
df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("fhvhv_tripdata_2021-01.csv.gz"))
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: integer (nullable = true)



In [6]:
#import pyspark
#from pyspark.sql import SparkSession
#spark = SparkSession.builder \
#    .master("local[*]") \
#    .appName('test') \
#    .getOrCreate()
#!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
#!gzip -dc fhvhv_tripdata_2021-01.csv.gz > fhvhv_tripdata_2021-01.csv
#!wc -l fhvhv_tripdata_2021-01.csv
#df = (spark.read
#    .option("header", "true")
#    .option("inferSchema", "true")
#    .csv("fhvhv_tripdata_2021-01.csv.gz"))
#df.printSchema()

In [5]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', IntegerType(), True)])

In [7]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [9]:
import pandas as pd

In [10]:
df_pandas = pd.read_csv('head.csv')

In [11]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [12]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

Integer - 4 bytes
Long - 8 bytes

In [13]:
from pyspark.sql import types

In [14]:
spark.createDataFrame(df_pandas).show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|    NaN|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|    NaN|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|    NaN|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|    NaN|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|    NaN|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [15]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [16]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [17]:
df = df.repartition(24)

In [19]:
df.head()


Row(hvfhs_license_num='HV0003', dispatching_base_num='B02765', pickup_datetime=datetime.datetime(2021, 1, 4, 10, 59, 8), dropoff_datetime=datetime.datetime(2021, 1, 4, 11, 6, 48), PULocationID=255, DOLocationID=112, SR_Flag=None)

In [20]:
df.write.parquet('fhvhv/2021/01/')

In [21]:
df = spark.read.parquet('fhvhv/2021/01/')

In [22]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [23]:
from pyspark.sql import functions as F

In [24]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-03 17:17:21|2021-01-03 17:26:18|         255|          34|   NULL|
|           HV0003|              B02882|2021-01-05 22:14:07|2021-01-05 22:32:28|         189|         107|   NULL|
|           HV0003|              B02867|2021-01-02 17:59:55|2021-01-02 18:10:39|          88|         137|   NULL|
|           HV0003|              B02872|2021-01-02 23:57:54|2021-01-03 00:15:48|         238|         224|   NULL|
|           HV0003|              B02875|2021-01-06 15:53:13|2021-01-06 16:07:07|         169|         208|   NULL|
|           HV0003|              B02867|2021-01-07 07:35:24|2021-01-07 07:55:49|

In [27]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [26]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID').show(5)

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-03 17:17:21|2021-01-03 17:26:18|         255|          34|
|2021-01-05 22:14:07|2021-01-05 22:32:28|         189|         107|
|2021-01-02 17:59:55|2021-01-02 18:10:39|          88|         137|
|2021-01-02 23:57:54|2021-01-03 00:15:48|         238|         224|
|2021-01-06 15:53:13|2021-01-06 16:07:07|         169|         208|
+-------------------+-------------------+------------+------------+
only showing top 5 rows


In [28]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(F.col('PULocationID') == 132) \
    .show(5)

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-06 19:23:50|2021-01-06 19:47:43|         132|          80|
|2021-01-06 15:19:18|2021-01-06 16:08:15|         132|          22|
|2021-01-02 13:15:42|2021-01-02 13:33:47|         132|         138|
|2021-01-01 11:02:45|2021-01-01 11:15:18|         132|         215|
|2021-01-01 15:06:15|2021-01-01 15:43:05|         132|         265|
+-------------------+-------------------+------------+------------+
only showing top 5 rows


In [29]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show(5)

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-05 22:14:07|2021-01-05 22:32:28|         189|         107|
|2021-01-02 17:59:55|2021-01-02 18:10:39|          88|         137|
|2021-01-02 23:57:54|2021-01-03 00:15:48|         238|         224|
|2021-01-06 15:53:13|2021-01-06 16:07:07|         169|         208|
|2021-01-07 07:35:24|2021-01-07 07:55:49|          75|          88|
+-------------------+-------------------+------------+------------+
only showing top 5 rows


In [30]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .take(5)

[Row(pickup_datetime=datetime.datetime(2021, 1, 5, 22, 14, 7), dropoff_datetime=datetime.datetime(2021, 1, 5, 22, 32, 28), PULocationID=189, DOLocationID=107),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 17, 59, 55), dropoff_datetime=datetime.datetime(2021, 1, 2, 18, 10, 39), PULocationID=88, DOLocationID=137),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 23, 57, 54), dropoff_datetime=datetime.datetime(2021, 1, 3, 0, 15, 48), PULocationID=238, DOLocationID=224),
 Row(pickup_datetime=datetime.datetime(2021, 1, 6, 15, 53, 13), dropoff_datetime=datetime.datetime(2021, 1, 6, 16, 7, 7), PULocationID=169, DOLocationID=208),
 Row(pickup_datetime=datetime.datetime(2021, 1, 7, 7, 35, 24), dropoff_datetime=datetime.datetime(2021, 1, 7, 7, 55, 49), PULocationID=75, DOLocationID=88)]

In [31]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .head(5)

[Row(pickup_datetime=datetime.datetime(2021, 1, 5, 22, 14, 7), dropoff_datetime=datetime.datetime(2021, 1, 5, 22, 32, 28), PULocationID=189, DOLocationID=107),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 17, 59, 55), dropoff_datetime=datetime.datetime(2021, 1, 2, 18, 10, 39), PULocationID=88, DOLocationID=137),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 23, 57, 54), dropoff_datetime=datetime.datetime(2021, 1, 3, 0, 15, 48), PULocationID=238, DOLocationID=224),
 Row(pickup_datetime=datetime.datetime(2021, 1, 6, 15, 53, 13), dropoff_datetime=datetime.datetime(2021, 1, 6, 16, 7, 7), PULocationID=169, DOLocationID=208),
 Row(pickup_datetime=datetime.datetime(2021, 1, 7, 7, 35, 24), dropoff_datetime=datetime.datetime(2021, 1, 7, 7, 55, 49), PULocationID=75, DOLocationID=88)]

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime','PULocationID','DOLocationID') \
    .filter(F.col('PULocationID') == 132) \
    .show(5)

In [33]:
df \
    .withColumn('pickup_date', F.to_date('pickup_datetime')) \
    .withColumn('dropoff_date', F.to_date('dropoff_datetime')) \
    .show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0005|              B02510|2021-01-03 17:17:21|2021-01-03 17:26:18|         255|          34|   NULL| 2021-01-03|  2021-01-03|
|           HV0003|              B02882|2021-01-05 22:14:07|2021-01-05 22:32:28|         189|         107|   NULL| 2021-01-05|  2021-01-05|
|           HV0003|              B02867|2021-01-02 17:59:55|2021-01-02 18:10:39|          88|         137|   NULL| 2021-01-02|  2021-01-02|
|           HV0003|              B02872|2021-01-02 23:57:54|2021-01-03 00:15:48|         238|         224|   NULL| 2021-01-02|  2021-01-03|
|           HV0003| 

In [34]:
df \
    .withColumn('pickup_date', F.to_date('pickup_datetime')) \
    .withColumn('dropoff_date', F.to_date('dropoff_datetime')) \
    .select('pickup_date', 'dropoff_date','PULocationID','DOLocationID') \
    .show(5)

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-03|  2021-01-03|         255|          34|
| 2021-01-05|  2021-01-05|         189|         107|
| 2021-01-02|  2021-01-02|          88|         137|
| 2021-01-02|  2021-01-03|         238|         224|
| 2021-01-06|  2021-01-06|         169|         208|
+-----------+------------+------------+------------+
only showing top 5 rows


In [35]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [36]:
crazy_stuff('B02884')

's/b44'

In [37]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [38]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-03|  2021-01-03|         255|          34|
|  e/b42| 2021-01-05|  2021-01-05|         189|         107|
|  e/b33| 2021-01-02|  2021-01-02|          88|         137|
|  e/b38| 2021-01-02|  2021-01-03|         238|         224|
|  e/b3b| 2021-01-06|  2021-01-06|         169|         208|
|  e/b33| 2021-01-07|  2021-01-07|          75|          88|
|  e/acc| 2021-01-07|  2021-01-07|         210|         210|
|  e/acc| 2021-01-02|  2021-01-02|         243|          69|
|  e/b35| 2021-01-04|  2021-01-04|         250|         213|
|  s/b3d| 2021-01-03|  2021-01-03|          87|          79|
|  e/a39| 2021-01-03|  2021-01-03|          68|         181|
|  s/acd| 2021-01-04|  2021-01-04|          95|         236|
|  s/b13| 2021-01-02|  2021-01-02|         262|         236|
|  e/9ce| 2021-01-04|  2

In [39]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')


DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [40]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
